# Benchmark: TabPFN vs. State-of-the-Art Algorithms

Comparison of **TabPFN** against standard ML baselines on the heart failure prediction task (3-class: early / late / healthy) using identical data pipeline (`load_final_data` → `preprocess_data` → `balance_data`).

### Algorithms
| Model | Type | Why |
|-------|------|-----|
| DummyClassifier | Baseline | Lower bound — shows what random guessing achieves |
| LogisticRegression | Linear | Classic baseline, interpretable, fast |
| RandomForest | Ensemble (Bagging) | Strong default, handles categoricals via encoding |
| XGBoost | Ensemble (Boosting) | State-of-the-art for tabular data |
| TabPFN | Foundation Model | Our main model — zero-shot Bayesian inference |

### Evaluation
- **Metrics**: Accuracy, F1 Macro, ROC-AUC (OvR), per-class F1
- **Robustness**: Each model trained on 20 balanced subsets (same seeds as TabPFN_v4)
- **Fair comparison**: Same train/val/test split, same features, same balancing

In [1]:
# Imports
import sys
sys.path.insert(0, '..')

import time
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, recall_score, precision_score,
    classification_report
)

from fs_thesis.data_loader import load_final_data
from fs_thesis.preprocessing import preprocess_data, balance_data, get_X_y

warnings.filterwarnings('ignore')

In [2]:
# ── Run-Ordner ──
_run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path(f"/Users/andrey/Repositories/fs-thesis/models/runs/benchmark_{_run_timestamp}")
PLOTS_DIR = RUN_DIR / "plots"
RESULTS_DIR = RUN_DIR / "results"
for d in [PLOTS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

_plot_counter = 0

def show_and_save(fig, name: str = None, width=1600, height=600):
    """fig.show() + PNG speichern."""
    global _plot_counter
    _plot_counter += 1
    filename = name or f"plot_{_plot_counter:02d}"
    path = PLOTS_DIR / f"{filename}.png"
    fig.write_image(str(path), scale=2, width=width, height=height)
    print(f"💾 {path}")
    fig.show()

print(f"📁 Run-Ordner: {RUN_DIR}")

📁 Run-Ordner: /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722


In [3]:
import logging

log_file = RUN_DIR / "run.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)
log = logging.getLogger()
log.info(f"📁 Run-Ordner: {RUN_DIR}")


22:17:22 | 📁 Run-Ordner: /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722


# 1. Data Pipeline (identical to TabPFN_v4)

In [4]:
from sklearn.model_selection import train_test_split

df = load_final_data()
df_train, df_val, df_test = preprocess_data(df)
X_val, y_val = get_X_y(df_val)
X_test, y_test = get_X_y(df_test)

_, X_val_small, _, y_val_small = train_test_split(
    X_val, y_val, test_size=3000, stratify=y_val, random_state=42
)
X_val_small = X_val_small.reset_index(drop=True)

log.info(f"Val: {len(y_val)} | Val (TabPFN subsample): {len(y_val_small)} | Test: {len(y_test)}")
log.info(f"Class distribution (val):       {np.bincount(y_val)}")
log.info(f"Class distribution (val_small): {np.bincount(y_val_small)}")

22:17:23 | Val: 35753 | Val (TabPFN subsample): 3000 | Test: 44691
22:17:23 | Class distribution (val):       [ 1719  1304 32730]
22:17:23 | Class distribution (val_small): [ 144  110 2746]


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)


# 2. Define Models

All models use the **same interface**: `fit(X_train, y_train)` → `predict(X_val)` / `predict_proba(X_val)`.

For tree-based and linear models, categorical features are **one-hot encoded** (TabPFN handles them natively). The encoding is applied inside the benchmark loop.

In [5]:
# Model Definitions
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

# Feature Types
FEATURE_COLS = [
    "gender", "anchor_age", "insurance", "language", "marital_status", "race", "admission_type", "bmi"
]
CAT_COLS = [
    "gender", "insurance", "language", "marital_status", "race", "admission_type"
]
NUM_COLS = ["anchor_age", "bmi"]

# Preprocessing Pipeline for sklearn Models
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), NUM_COLS),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CAT_COLS),
    ],
    remainder='drop'
)

# Model Dictionary
MODELS = {
    "DummyClassifier": Pipeline([
        ('prep', preprocessor),
        ('clf', DummyClassifier(strategy='stratified', random_state=42))
    ]),
    "LogisticRegression": Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42))
    ]),
    "RandomForest": Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1))
    ]),
    "XGBoost": Pipeline([
        ('prep', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            objective='multi:softprob', num_class=3,
            random_state=42, n_jobs=-1, verbosity=0,
            eval_metric='mlogloss'
        ))
    ]),
    #"TabPFN": None (extra in cell 10)
}

log.info(f"Models: {list(MODELS.keys())}")
log.info(f"Features: {len(FEATURE_COLS)} ({len(NUM_COLS)} numerical, {len(CAT_COLS)} categorical)")

22:17:23 | Models: ['DummyClassifier', 'LogisticRegression', 'RandomForest', 'XGBoost']
22:17:23 | Features: 8 (2 numerical, 6 categorical)


In [6]:
log.info(len(y_val))

22:17:23 | 35753


# 3. Robustness Benchmark Loop

Each model is trained **20 times** with different balanced training subsets (seeds 42–61), identical to the TabPFN_v4 robustness loop. This measures performance **and** stability.

In [7]:
from sklearn.base import clone

N_LOOPS = 20
N_SAMPLES = 300
log.info(f"RUN gestartet | N_LOOPS={N_LOOPS} | N_SAMPLES={N_SAMPLES}")

config = {"n_loops": N_LOOPS, "n_samples": N_SAMPLES, 
          "models": list(MODELS.keys()) + ["TabPFN", "TabICL"],  # ← TabICL hinzufügen
          "run_dir": str(RUN_DIR)}
json.dump(config, open(RUN_DIR / "config.json", "w"), indent=2)

all_results = []
start_total = time.time()

22:17:23 | RUN gestartet | N_LOOPS=20 | N_SAMPLES=300


In [8]:
import subprocess, json
tabicl_result_path = str(RESULTS_DIR / "tabicl_result.json")

log.info("Starte TabICL subprocess...")
proc_icl = subprocess.run(
    [sys.executable, "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabicl.py", tabicl_result_path],
    capture_output=False
)

if proc_icl.returncode == 0:
    tabicl_result = json.load(open(tabicl_result_path))
    all_results.append(tabicl_result)
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)
    log.info(f"✅ TabICL: F1={tabicl_result['f1_macro']:.4f}")
else:
    log.error("❌ TabICL subprocess fehlgeschlagen")

22:17:23 | Starte TabICL subprocess...


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
predicting...
proba...
✅ F1=0.3637 | AUC=0.7893 | 12.5s


22:17:39 | ✅ TabICL: F1=0.3637


In [9]:


tabpfn_result_path = str(RESULTS_DIR / "tabpfn_result.json")

log.info("Starte TabPFN subprocess (isoliert von MPS)...")
proc = subprocess.run(
    [sys.executable, "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabpfn.py", tabpfn_result_path],
    capture_output=False
)

if proc.returncode == 0:
    tabpfn_result = json.load(open(tabpfn_result_path))
    all_results.append(tabpfn_result)
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)
    log.info(f"✅ TabPFN geladen: F1={tabpfn_result['f1_macro']:.4f}")
else:
    log.error("❌ TabPFN subprocess fehlgeschlagen")

22:17:39 | Starte TabPFN subprocess (isoliert von MPS)...


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
predicting...
proba...
✅ F1=0.3684 | AUC=0.8145 | 21.7s


22:18:05 | ✅ TabPFN geladen: F1=0.3684


In [10]:
from sklearn.base import clone

for model_name, pipeline in MODELS.items():
    log.info(f"\n{'='*60}")
    log.info(f"  {model_name}")
    log.info(f"{'='*60}")
    t0 = time.time()
    model_results = []

    for i in tqdm(range(N_LOOPS), desc=model_name):
        try:
            seed = 42 + i
            df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=seed)
            X_tr, y_tr = get_X_y(df_bal)
            clf = clone(pipeline)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_val)
            y_proba = clf.predict_proba(X_val)
            f1_pc = f1_score(y_val, y_pred, average=None)
            result = {
                'model': model_name, 'run_id': i, 'seed': seed,
                'accuracy': accuracy_score(y_val, y_pred),
                'f1_macro': f1_score(y_val, y_pred, average='macro'),
                'roc_auc_macro': roc_auc_score(y_val, y_proba, multi_class='ovr', average='macro'),
                'recall_macro': recall_score(y_val, y_pred, average='macro'),
                'precision_macro': precision_score(y_val, y_pred, average='macro'),
                'f1_class_0_early': f1_pc[0],
                'f1_class_1_late': f1_pc[1],
                'f1_class_2_healthy': f1_pc[2],
                'time_sec': time.time() - t0
            }
            all_results.append(result)
            model_results.append(result)
        except Exception as e:
            log.error(f"  ⚠️ ERROR Run {i}: {e}")

    elapsed = time.time() - t0
    if model_results:
        df_m = pd.DataFrame(model_results)
        log.info(f"  ✅ F1={df_m['f1_macro'].mean():.4f}±{df_m['f1_macro'].std():.4f} | AUC={df_m['roc_auc_macro'].mean():.4f} | {elapsed:.1f}s")
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)

log.info(f"\n{'='*60}")
log.info(f"  DONE | {len(all_results)} total runs")
log.info(f"{'='*60}")

22:18:05 | 
22:18:05 |   DummyClassifier
22:18:05 | ============================================================


DummyClassifier:   0%|          | 0/20 [00:00<?, ?it/s]

22:18:07 |   ✅ F1=0.2106±0.0000 | AUC=0.4946 | 2.3s
22:18:07 | 
22:18:07 |   LogisticRegression
22:18:07 | ============================================================


LogisticRegression:   0%|          | 0/20 [00:00<?, ?it/s]

22:18:10 |   ✅ F1=0.3615±0.0047 | AUC=0.7635 | 2.5s
22:18:10 | 
22:18:10 |   RandomForest
22:18:10 | ============================================================


RandomForest:   0%|          | 0/20 [00:00<?, ?it/s]

22:18:17 |   ✅ F1=0.3664±0.0064 | AUC=0.7881 | 7.4s
22:18:17 | 
22:18:17 |   XGBoost
22:18:17 | ============================================================


XGBoost:   0%|          | 0/20 [00:00<?, ?it/s]

22:18:33 |   ✅ F1=0.3607±0.0062 | AUC=0.7721 | 16.0s
22:18:33 | 
22:18:33 |   DONE | 82 total runs
22:18:33 | ============================================================


# 4. Results & Comparison

In [11]:
# ── Summary Table ──
df_results = pd.DataFrame(all_results)
df_results.to_csv(RESULTS_DIR / "benchmark_all_runs.csv", index=False)

df_summary = df_results.groupby('model').agg(
    f1_mean=('f1_macro', 'mean'), f1_std=('f1_macro', 'std'),
    auc_mean=('roc_auc_macro', 'mean'), auc_std=('roc_auc_macro', 'std'),
    acc_mean=('accuracy', 'mean'), acc_std=('accuracy', 'std'),
    recall_mean=('recall_macro', 'mean'),
    precision_mean=('precision_macro', 'mean'),
    f1_early_mean=('f1_class_0_early', 'mean'), f1_early_std=('f1_class_0_early', 'std'),
    f1_late_mean=('f1_class_1_late', 'mean'), f1_late_std=('f1_class_1_late', 'std'),
    f1_healthy_mean=('f1_class_2_healthy', 'mean'), f1_healthy_std=('f1_class_2_healthy', 'std'),
    n_runs=('run_id', 'count'),
).reset_index().sort_values('f1_mean', ascending=False)

df_summary.to_csv(RESULTS_DIR / "benchmark_summary.csv", index=False)

# Schöne Darstellung
log.info("\n📊 Benchmark Summary (sorted by F1 Macro):\n")
display_cols = ['model', 'f1_mean', 'f1_std', 'auc_mean', 'auc_std', 'acc_mean', 'recall_mean', 'precision_mean']
log.info(df_summary[display_cols].to_string(index=False, float_format='{:.4f}'.format))

22:18:33 | 
📊 Benchmark Summary (sorted by F1 Macro):

22:18:33 |              model  f1_mean  f1_std  auc_mean  auc_std  acc_mean  recall_mean  precision_mean
            TabPFN   0.3684     NaN    0.8145      NaN    0.5560       0.6472          0.4081
      RandomForest   0.3664  0.0064    0.7881   0.0041    0.5658       0.6166          0.4019
            TabICL   0.3637     NaN    0.7893      NaN    0.5473       0.6353          0.4037
LogisticRegression   0.3615  0.0047    0.7635   0.0044    0.5639       0.5771          0.3992
           XGBoost   0.3607  0.0062    0.7721   0.0052    0.5643       0.5971          0.3971
   DummyClassifier   0.2106  0.0000    0.4946   0.0000    0.3305       0.3274          0.3315


## 4.1 F1 Macro Comparison

In [12]:
# F1 Macro: Bar Chart with Error Bars
model_order = df_summary.sort_values('f1_mean')['model'].tolist()

fig = px.bar(
    df_summary.sort_values('f1_mean'),
    x='f1_mean', y='model', error_x='f1_std',
    orientation='h',
    text=df_summary.sort_values('f1_mean').apply(
        lambda r: f"{r['f1_mean']:.1%} ± {r['f1_std']:.1%}", axis=1
    ),
    title=f'Benchmark: F1 Macro ({N_LOOPS} Runs, n_samples={N_SAMPLES})',
    labels={'f1_mean': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white'
)
fig.update_layout(
    xaxis_tickformat='.0%',
    showlegend=False,
    yaxis=dict(categoryorder='array', categoryarray=model_order),
    height=400
)
show_and_save(fig, "benchmark_f1_macro_comparison")

22:18:34 | Chromium init'ed with kwargs {}
22:18:34 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:18:34 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp6gna9n02.
22:18:34 | Opening browser.
22:18:34 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmplj0_h21t.
22:18:34 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmplj0_h21t
22:18:34 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp6gna9n02/index.html
22:18:34 | Waiting on all navigates
22:18:34 | All navigates done, putting them all in queue.
22:18:35 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp6gna9n02/index.html
22:18:35 | Waiting on all navigates
22:18:35 | All navigates done, putting them all in queue.
22:18:35 | Tab ready: 602FDC6EAC7D2DA764FB6C6957AD9CEC
22:18:35 | Getting tab from queue (has 1)
22:18:35 | Got 602F
22:18:35 | Processing Benchma

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/benchmark_f1_macro_comparison.png


In [13]:
# F1 Macro: Violin Plot (Distribution across Runs)
fig2 = px.violin(
    df_results, x='model', y='f1_macro', box=True, points='all',
    title=f'F1 Macro Distribution ({N_LOOPS} Runs per Model)',
    labels={'f1_macro': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white',
    category_orders={'model': model_order[::-1]}
)
fig2.update_layout(yaxis_tickformat='.0%', showlegend=False, height=500)
show_and_save(fig2, "benchmark_f1_violin")

22:18:36 | TemporaryDirectory.cleanup() worked.
22:18:36 | shutil.rmtree worked.
22:18:36 | Chromium init'ed with kwargs {}
22:18:36 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:18:36 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmptdl2zgy5.
22:18:36 | Opening browser.
22:18:36 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpdlxixi0d.
22:18:36 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpdlxixi0d
22:18:36 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmptdl2zgy5/index.html
22:18:36 | Waiting on all navigates
22:18:36 | All navigates done, putting them all in queue.
22:18:36 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmptdl2zgy5/index.html
22:18:36 | Waiting on all navigates
22:18:37 | All navigates done, putting them all in queue.
22:18:37 | Tab ready: 9C2DFC81019928538059242F5FF83D89
22:18:37 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/benchmark_f1_violin.png


## 4.2 ROC-AUC Comparison

In [14]:
# AUC + F1 Combined: Grouped Bar
metrics_long = []
for _, row in df_summary.iterrows():
    metrics_long.append({'model': row['model'], 'Metric': 'F1 Macro', 'Score': row['f1_mean'], 'Std': row['f1_std']})
    metrics_long.append({'model': row['model'], 'Metric': 'ROC-AUC', 'Score': row['auc_mean'], 'Std': row['auc_std']})
    metrics_long.append({'model': row['model'], 'Metric': 'Accuracy', 'Score': row['acc_mean'], 'Std': row['acc_std']})

ml_df = pd.DataFrame(metrics_long)

fig3 = px.bar(
    ml_df, x='model', y='Score', color='Metric', error_y='Std',
    barmode='group',
    title=f'Benchmark: All Metrics ({N_LOOPS} Runs)',
    template='plotly_white',
    text_auto='.1%',
    color_discrete_map={'F1 Macro': '#e74c3c', 'ROC-AUC': '#3498db', 'Accuracy': '#2ecc71'},
    category_orders={'model': model_order[::-1]}
)
fig3.update_layout(yaxis_tickformat='.0%', xaxis_title=None, height=500)
show_and_save(fig3, "benchmark_all_metrics")

22:18:37 | TemporaryDirectory.cleanup() worked.
22:18:37 | shutil.rmtree worked.
22:18:37 | Chromium init'ed with kwargs {}
22:18:37 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:18:37 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmps36mqta_.
22:18:37 | Opening browser.
22:18:37 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpplzfchn5.
22:18:37 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpplzfchn5
22:18:38 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmps36mqta_/index.html
22:18:38 | Waiting on all navigates
22:18:38 | All navigates done, putting them all in queue.
22:18:38 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmps36mqta_/index.html
22:18:38 | Waiting on all navigates
22:18:39 | All navigates done, putting them all in queue.
22:18:39 | Tab ready: DAF529FF052A2FD887133B47B4AE18D1
22:18:39 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/benchmark_all_metrics.png


## 4.3 Per-Class F1 Comparison

In [15]:
# Per-Class F1: Heatmap
class_cols = {
    'Early (<1Y)': 'f1_early_mean',
    'Late (>1Y)': 'f1_late_mean',
    'Healthy': 'f1_healthy_mean'
}

# Build matrix
heatmap_data = []
models_sorted = df_summary.sort_values('f1_mean', ascending=False)['model'].tolist()

for model in models_sorted:
    row = df_summary[df_summary['model'] == model].iloc[0]
    heatmap_data.append([row[col] for col in class_cols.values()])

z = np.array(heatmap_data)
annot = [[f"{v:.1%}" for v in row] for row in z]

fig4 = ff.create_annotated_heatmap(
    z,
    x=list(class_cols.keys()),
    y=models_sorted,
    annotation_text=annot,
    colorscale='RdYlGn',
    showscale=True
)
fig4.update_layout(
    title='Per-Class F1 Score by Model',
    template='plotly_white',
    height=400
)
show_and_save(fig4, "benchmark_per_class_heatmap")

22:18:40 | TemporaryDirectory.cleanup() worked.
22:18:40 | shutil.rmtree worked.
22:18:40 | Chromium init'ed with kwargs {}
22:18:40 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:18:40 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmprmev2qfa.
22:18:40 | Opening browser.
22:18:40 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp4appg0jx.
22:18:40 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp4appg0jx
22:18:40 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmprmev2qfa/index.html
22:18:40 | Waiting on all navigates
22:18:40 | All navigates done, putting them all in queue.
22:18:40 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmprmev2qfa/index.html
22:18:40 | Waiting on all navigates
22:18:41 | All navigates done, putting them all in queue.
22:18:41 | Tab ready: 0EAFB1579C5344477B32977045FF1523
22:18:41 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/benchmark_per_class_heatmap.png


# 5. Final Test Evaluation

Best model per algorithm (highest F1 on val) evaluated once on the **test set**.

In [16]:
from sklearn.base import clone

test_results = []

# sklearn Modelle
for model_name, pipeline in MODELS.items():
    df_model = df_results[df_results['model'] == model_name]
    best_row = df_model.loc[df_model['f1_macro'].idxmax()]
    best_seed = int(best_row['seed'])

    df_bal_best = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr_best, y_tr_best = get_X_y(df_bal_best)
    clf_test = clone(MODELS[model_name])
    clf_test.fit(X_tr_best, y_tr_best)
    y_test_pred = clf_test.predict(X_test)
    y_test_proba = clf_test.predict_proba(X_test)

    f1_pc = f1_score(y_test, y_test_pred, average=None)
    test_results.append({
        'model': model_name, 'best_seed': best_seed,
        'val_f1': best_row['f1_macro'],
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'test_f1_macro': f1_score(y_test, y_test_pred, average='macro'),
        'test_roc_auc': roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro'),
        'test_f1_early': f1_pc[0], 'test_f1_late': f1_pc[1], 'test_f1_healthy': f1_pc[2],
    })
    log.info(f"\n📊 {model_name} (seed={best_seed}):")
    log.info(f"   Test: Acc={test_results[-1]['test_accuracy']:.2%} | F1={test_results[-1]['test_f1_macro']:.2%} | AUC={test_results[-1]['test_roc_auc']:.2%}")
    log.info(classification_report(y_test, y_test_pred, target_names=['Early (<1Y)', 'Late (>1Y)', 'Healthy']))

# TabPFN via subprocess
tabpfn_val_seed = int(df_results[df_results['model'] == 'TabPFN'].iloc[0]['seed'])
tabpfn_test_path = str(RESULTS_DIR / "tabpfn_test_result.json")
log.info(f"Starte TabPFN Test subprocess (seed={tabpfn_val_seed})...")
proc = subprocess.run([
    sys.executable,
    "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabpfn_test.py",
    tabpfn_test_path, str(tabpfn_val_seed), str(N_SAMPLES)
])
if proc.returncode == 0:
    tabpfn_test = json.load(open(tabpfn_test_path))
    tabpfn_test['val_f1'] = float(df_results[df_results['model'] == 'TabPFN'].iloc[0]['f1_macro'])
    test_results.append(tabpfn_test)
    log.info(f"\n📊 TabPFN (seed={tabpfn_val_seed}):")
    log.info(f"   Test: F1={tabpfn_test['test_f1_macro']:.2%} | AUC={tabpfn_test['test_roc_auc']:.2%}")
else:
    log.error("❌ TabPFN Test subprocess error")

# TabICL via subprocess
tabicl_test_path = str(RESULTS_DIR / "tabicl_test_result.json")
log.info(f"Starte TabICL Test subprocess...")
proc_icl = subprocess.run([
    sys.executable,
    "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabicl_test.py",
    tabicl_test_path, str(42), str(N_SAMPLES)
])
if proc_icl.returncode == 0:
    tabicl_test = json.load(open(tabicl_test_path))
    tabicl_test['val_f1'] = float(df_results[df_results['model'] == 'TabICL'].iloc[0]['f1_macro'])
    test_results.append(tabicl_test)
    log.info(f"\n📊 TabICL: F1={tabicl_test['test_f1_macro']:.2%} | AUC={tabicl_test['test_roc_auc']:.2%}")
else:
    log.error("❌ TabICL Test subprocess error")

df_test_results = pd.DataFrame(test_results).sort_values('test_f1_macro', ascending=False)
df_test_results.to_csv(RESULTS_DIR / "test_final_results.csv", index=False)
log.info("\n" + df_test_results.to_string(index=False))

22:18:41 | TemporaryDirectory.cleanup() worked.
22:18:41 | shutil.rmtree worked.
22:18:41 | 
📊 DummyClassifier (seed=42):
22:18:41 |    Test: Acc=33.18% | F1=21.25% | AUC=50.07%
22:18:41 |               precision    recall  f1-score   support

 Early (<1Y)       0.05      0.33      0.08      2148
  Late (>1Y)       0.04      0.34      0.07      1630
     Healthy       0.91      0.33      0.49     40913

    accuracy                           0.33     44691
   macro avg       0.33      0.34      0.21     44691
weighted avg       0.84      0.33      0.45     44691

22:18:41 | 
📊 LogisticRegression (seed=44):
22:18:41 |    Test: Acc=58.04% | F1=37.25% | AUC=76.74%
22:18:41 |               precision    recall  f1-score   support

 Early (<1Y)       0.17      0.59      0.26      2148
  Late (>1Y)       0.08      0.58      0.13      1630
     Healthy       0.97      0.58      0.73     40913

    accuracy                           0.58     44691
   macro avg       0.40      0.58      0.37    

Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
predicting test...
proba test...
✅ F1=0.3607 | AUC=0.8003


22:24:12 | 
📊 TabPFN (seed=42):
22:24:12 |    Test: F1=36.07% | AUC=80.03%
22:24:12 | Starte TabICL Test subprocess...


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
predicting test...
proba test...
✅ F1=0.3691 | AUC=0.8075


22:27:57 | 
📊 TabICL: F1=36.91% | AUC=80.75%
22:27:57 | 
             model  best_seed   val_f1  test_accuracy  test_f1_macro  test_roc_auc  test_f1_early  test_f1_late  test_f1_healthy                                                                                                      proba_path
      RandomForest         44 0.384043       0.609026       0.385679      0.788576       0.242400      0.165437         0.749200                                                                                                             NaN
           XGBoost         44 0.374683       0.600434       0.378918      0.783883       0.223853      0.171018         0.741884                                                                                                             NaN
LogisticRegression         44 0.369503       0.580385       0.372472      0.767431       0.257827      0.133193         0.726396                                                                                            

In [17]:
# --- Test Results: Bar Chart  ---
df_test_sorted = df_test_results.sort_values('test_f1_macro')

fig5 = px.bar(
    df_test_sorted,
    x='test_f1_macro', y='model',
    orientation='h',
    text=df_test_sorted.apply(
        lambda r: f"F1={r['test_f1_macro']:.1%} | AUC={r['test_roc_auc']:.1%}", axis=1
    ),
    title='Final Test Set: F1 Macro by Model',
    labels={'test_f1_macro': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white'
)
fig5.update_layout(xaxis_tickformat='.0%', showlegend=False, height=400)
show_and_save(fig5, "benchmark_test_f1")

22:27:57 | Chromium init'ed with kwargs {}
22:27:57 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:27:57 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpu6_9r11u.
22:27:57 | Opening browser.
22:27:57 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpwo5zudlt.
22:27:57 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpwo5zudlt
22:27:58 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpu6_9r11u/index.html
22:27:58 | Waiting on all navigates
22:27:58 | All navigates done, putting them all in queue.
22:27:58 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpu6_9r11u/index.html
22:27:58 | Waiting on all navigates
22:27:59 | All navigates done, putting them all in queue.
22:27:59 | Tab ready: DFE15CD844AB5CF60F55DECF99BD983D
22:27:59 | Getting tab from queue (has 1)
22:27:59 | Got DFE1
22:27:59 | Processing Final_T

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/benchmark_test_f1.png


# 6. ROC Curves (Test Set, Best Models)

One-vs-Rest ROC curves for each model on the test set, all in one plot per class.

In [18]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.base import clone
from pathlib import Path

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
class_names = ['Early (<1Y)', 'Late (>1Y)', 'Healthy']
model_colors = {
    'DummyClassifier': '#95a5a6',
    'LogisticRegression': '#3498db',
    'RandomForest': '#2ecc71',
    'XGBoost': '#e67e22',
    'TabPFN': '#e74c3c',
    'TabICL': '#9b59b6',
}

# sklearn Modelle neu trainieren
best_probas = {}
for model_name, pipeline in MODELS.items():
    df_model = df_results[df_results['model'] == model_name]
    best_seed = int(df_model.loc[df_model['f1_macro'].idxmax(), 'seed'])
    df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr, y_tr = get_X_y(df_bal)
    clf = clone(MODELS[model_name])
    clf.fit(X_tr, y_tr)
    best_probas[model_name] = clf.predict_proba(X_test)

# TabPFN proba aus Subprocess-Datei laden
tabpfn_proba_path = tabpfn_test_path.replace('.json', '_proba.npy')
if Path(tabpfn_proba_path).exists():
    best_probas['TabPFN'] = np.load(tabpfn_proba_path)
    log.info("TabPFN proba loaded")
else:
    log.warning("⚠️ TabPFN not found")

tabicl_proba_path = tabicl_test_path.replace('.json', '_proba.npy')
if Path(tabicl_proba_path).exists():
    best_probas['TabICL'] = np.load(tabicl_proba_path)
    log.info("TabICL proba loaded")
else:
    log.warning("⚠️ TabICL not found")

# Plot
from plotly.subplots import make_subplots

fig6 = make_subplots(rows=1, cols=3, subplot_titles=[f'OvR: {cn}' for cn in class_names])

for i, cn in enumerate(class_names):
    for model_name, proba in best_probas.items():
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba[:, i])
        roc_auc_val = auc(fpr, tpr)
        fig6.add_trace(
            go.Scatter(x=fpr, y=tpr, mode='lines',
                       name=f'{model_name} ({roc_auc_val:.3f})',
                       line=dict(color=model_colors.get(model_name, 'grey'), width=2),
                       showlegend=(i == 0)),
            row=1, col=i+1
        )
    fig6.add_trace(
        go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                   line=dict(color='grey', width=1, dash='dash'), showlegend=False),
        row=1, col=i+1
    )

fig6.update_layout(
    title='ROC Curves (Test Set) — One-vs-Rest per Class',
    template='plotly_white', height=600, width=1600,
    legend=dict(x=1.02, y=1)
)
for i in range(3):
    fig6.update_xaxes(title_text='FPR', row=1, col=i+1)
    fig6.update_yaxes(title_text='TPR', row=1, col=i+1)

show_and_save(fig6, "benchmark_roc_curves", width=1600, height=600)

22:27:59 | TemporaryDirectory.cleanup() worked.
22:27:59 | shutil.rmtree worked.
22:28:00 | TabPFN proba loaded
22:28:00 | TabICL proba loaded
22:28:00 | Chromium init'ed with kwargs {}
22:28:00 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:28:00 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpwb6cd6pr.
22:28:00 | Opening browser.
22:28:00 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpw4hapf0h.
22:28:00 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpw4hapf0h
22:28:01 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpwb6cd6pr/index.html
22:28:01 | Waiting on all navigates
22:28:01 | All navigates done, putting them all in queue.
22:28:01 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpwb6cd6pr/index.html
22:28:01 | Waiting on all navigates
22:28:01 | All navigates done, putting them all in queue.
22:

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/benchmark_roc_curves.png


# 7. Data Quality Check: Dead but "Healthy"?

Patients classified as **target=2 (healthy/censored)** who have a recorded date of death (`dod`). These patients died but were never diagnosed with heart failure — they are **correctly** censored (not false labels), but it's important to know how many there are and whether they bias the model.

In [19]:
# ── Data Quality: Deceased Patients in the "Healthy" Class ──
import polars as pl

# df is the full dataset (before the split)
df_quality = df.to_pandas() if hasattr(df, 'to_pandas') else df

# Patients with target=2 (healthy/censored) AND date of death (dod) present
dead_but_healthy = df.filter(
    (pl.col("target") == 2) & (pl.col("dod").is_not_null())
)

total_healthy = df.filter(pl.col("target") == 2).height
n_dead_healthy = dead_but_healthy.height

log.info(f"Total patients with target=2 (healthy/censored): {total_healthy:,}")
log.info(f"With date of death (dod): {n_dead_healthy:,} ({n_dead_healthy/total_healthy:.1%})")
log.info(f"Without dod (truly alive): {total_healthy - n_dead_healthy:,}")

# Distribution of survival time for the 'dead healthy' patients
if n_dead_healthy > 0:
    t_death_stats = dead_but_healthy.select("t_death").to_pandas()["t_death"].describe()
    log.info(f"\nSurvival time (t_death) of deceased 'healthy' patients:")
    log.info(t_death_stats)
    
    fig_dq = px.histogram(
        dead_but_healthy.select("t_death").to_pandas(), 
        x="t_death", nbins=50,
        title=f"Deceased patients without HF diagnosis (n={n_dead_healthy:,}): Days until death",
        labels={"t_death": "Days from baseline to death"},
        template="plotly_white"
    )
    fig_dq.add_vline(x=365, line_dash="dash", line_color="red", annotation_text="1 year")
    show_and_save(fig_dq, "data_quality_dead_but_healthy")

22:28:03 | Total patients with target=2 (healthy/censored): 204,562
22:28:03 | With date of death (dod): 29,503 (14.4%)
22:28:03 | Without dod (truly alive): 175,059
22:28:03 | 
Survival time (t_death) of deceased 'healthy' patients:
22:28:03 | count    29503.000000
mean       651.097956
std        966.801290
min      -2520.000000
25%         39.000000
50%        217.000000
75%        845.000000
max       5615.000000
Name: t_death, dtype: float64
22:28:03 | Chromium init'ed with kwargs {}
22:28:03 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:28:03 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmphrfpkx7r.
22:28:03 | Opening browser.
22:28:03 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpk7l3p6xt.
22:28:03 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpk7l3p6xt
22:28:03 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmphrfpkx7r/

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/data_quality_dead_but_healthy.png


# 8. Correlation vs. Causality (Feature Analysis)

**Philipp's feedback: "Remove confounders"** — Identify features with correlation but no causality.

| Analysis | What it measures | Method |
|----------|------------------|--------|
| **Correlation** (univariate) | How strongly does a feature *alone* relate to the target? | Cramér's V (categorical), Eta² (numerical) |
| **Predictive importance** (multivariate) | How much *unique* predictive power does a feature have, when all others are known? | Permutation Importance (F1 Macro) |

**Interpretation:**
- High correlation + high importance → **True driver** (e.g., age, BMI)
- High correlation + low/negative importance → **Confounder** (e.g., insurance correlates with age)
- Low correlation + low importance → **Irrelevant** (can be removed)


## 8.1 Univariate Correlation (Feature ↔ Target)

In [20]:
# Univariate Correlation: How strongly does each feature alone relate to the target?
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    """Cramér's V: Association for categorical × categorical (0 = none, 1 = perfect)."""
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = len(x)
    min_dim = min(ct.shape) - 1
    if min_dim == 0 or n == 0:
        return 0.0
    return np.sqrt(chi2 / (n * min_dim))

def eta_squared(feature_values, target_values):
    """Eta²: Effect size for numerical × categorical (ANOVA). 0 = none, 1 = perfect."""
    groups = [feature_values[target_values == c].dropna() for c in sorted(target_values.unique())]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return 0.0
   
    grand_mean = feature_values.dropna().mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total = ((feature_values.dropna() - grand_mean)**2).sum()
    if ss_total == 0:
        return 0.0
    return ss_between / ss_total

# Compute correlation for each feature
df_val_pd = X_val.copy()
df_val_pd['target'] = y_val

correlation_results = []
for feature in FEATURE_COLS:
    if feature in CAT_COLS:
        corr = cramers_v(df_val_pd[feature], df_val_pd['target'])
        method = "Cramér's V"
    else:
        corr = eta_squared(df_val_pd[feature], df_val_pd['target'])
        method = "Eta²"
    correlation_results.append({
        'Feature': feature,
        'Correlation': corr,
        'Method': method,
        'Type': 'categorical' if feature in CAT_COLS else 'numerical'
    })

df_corr = pd.DataFrame(correlation_results).sort_values('Correlation', ascending=False)

log.info("\nUnivariate Correlation (Feature → Target):\n")
log.info(df_corr.to_string(index=False, float_format='{:.4f}'.format))

# Plot
fig_corr = px.bar(
    df_corr.sort_values('Correlation'),
    x='Correlation', y='Feature', orientation='h',
    color='Type',
    color_discrete_map={'categorical': '#3498db', 'numerical': '#e74c3c'},
    text=df_corr.sort_values('Correlation').apply(
        lambda r: f"{r['Correlation']:.3f} ({r['Method']})", axis=1
    ),
    title=f"Univariate Correlation: Feature ↔ Target (n={len(y_val)})",
    labels={'Correlation': 'Association Strength', 'Feature': ''},
    template='plotly_white'
)
fig_corr.update_layout(height=450)
show_and_save(fig_corr, "correlation_univariate")

22:28:05 | 
Univariate Correlation (Feature → Target):

22:28:05 |        Feature  Correlation     Method        Type
admission_type       0.1410 Cramér's V categorical
     insurance       0.1235 Cramér's V categorical
          race       0.0843 Cramér's V categorical
marital_status       0.0665 Cramér's V categorical
      language       0.0534 Cramér's V categorical
    anchor_age       0.0483       Eta²   numerical
        gender       0.0395 Cramér's V categorical
           bmi       0.0089       Eta²   numerical
22:28:05 | Chromium init'ed with kwargs {}
22:28:05 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:28:05 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmprq57q8p7.
22:28:05 | Opening browser.
22:28:05 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpz17psnik.
22:28:05 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpz17psnik
22:28:05 | Conformin

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/correlation_univariate.png


## 8.2 Predictive Importance (Permutation Importance)

Measures how much **unique** predictive power a feature has **when all other features are known**. A feature with high correlation but low importance is a confounder — it only correlates because it is associated with a true driver.

In [21]:
from sklearn.inspection import permutation_importance
from sklearn.base import clone

importance_results = []

for model_name in MODELS.keys():
    log.info(f"\n🔄 {model_name}...")
    df_model = df_results[df_results['model'] == model_name]
    best_seed = int(df_model.loc[df_model['f1_macro'].idxmax(), 'seed'])

    df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr, y_tr = get_X_y(df_bal)
    clf = clone(MODELS[model_name])
    clf.fit(X_tr, y_tr)
    perm = permutation_importance(clf, X_val, y_val, n_repeats=10,
                                   random_state=42, scoring='f1_macro', n_jobs=-1)
    for j, feature in enumerate(FEATURE_COLS):
        importance_results.append({
            'Model': model_name, 'Feature': feature,
            'Importance': perm.importances_mean[j],
            'Std': perm.importances_std[j],
        })
    log.info(f"  ✅ done")


log.info("TabPFN Permutation Importance skipped (35k Val × 10 repeats)")
for feature in FEATURE_COLS:
    importance_results.append({
        'Model': 'TabPFN', 'Feature': feature,
        'Importance': float('nan'), 'Std': float('nan')
    })

log.info("TabICL Permutation Importance skipped")
for feature in FEATURE_COLS:
    importance_results.append({
        'Model': 'TabICL', 'Feature': feature,
        'Importance': float('nan'), 'Std': float('nan')
    })

df_importance = pd.DataFrame(importance_results)
df_importance.to_csv(RESULTS_DIR / "permutation_importance_all_models.csv", index=False)

df_imp_avg = df_importance[
    ~df_importance['Model'].isin(['DummyClassifier', 'TabPFN', 'TabICL'])
].groupby('Feature').agg(
    Importance_mean=('Importance', 'mean'),
    Importance_std=('Importance', 'std'),
).reset_index().sort_values('Importance_mean', ascending=False)

log.info("\n📊 Average Permutation Importance (excluding Dummy + TabPFN):\n")
log.info(df_imp_avg.to_string(index=False, float_format='{:.4f}'.format))

22:28:06 | 
🔄 DummyClassifier...
22:28:10 |   ✅ done
22:28:10 | 
🔄 LogisticRegression...
22:28:11 |   ✅ done
22:28:11 | 
🔄 RandomForest...
22:28:11 | TemporaryDirectory.cleanup() worked.
22:28:11 | shutil.rmtree worked.
22:28:19 |   ✅ done
22:28:19 | 
🔄 XGBoost...
22:28:24 |   ✅ done
22:28:24 | TabPFN Permutation Importance skipped (35k Val × 10 repeats)
22:28:24 | TabICL Permutation Importance skipped
22:28:24 | 
📊 Average Permutation Importance (excluding Dummy + TabPFN):

22:28:24 |        Feature  Importance_mean  Importance_std
admission_type           0.0432          0.0036
    anchor_age           0.0382          0.0104
           bmi           0.0143          0.0107
     insurance           0.0037          0.0038
          race           0.0014          0.0017
        gender           0.0007          0.0010
      language           0.0003          0.0016
marital_status          -0.0003          0.0008


In [22]:
# ── Importance Heatmap: pro Modell × Feature ──
pivot = df_importance.pivot_table(index='Model', columns='Feature', values='Importance')
models_order = ['DummyClassifier', 'LogisticRegression', 'RandomForest', 'XGBoost', 'TabPFN', 'TabICL']
pivot = pivot.reindex(models_order)
features_order = df_imp_avg.sort_values('Importance_mean', ascending=True)['Feature'].tolist()
pivot = pivot[features_order]

annot = [[f"{v:.3f}" for v in row] for row in pivot.values]

fig_imp_heat = ff.create_annotated_heatmap(
    pivot.values, x=features_order, y=models_order,
    annotation_text=annot, colorscale='RdBu', reversescale=False, showscale=True
)
fig_imp_heat.update_layout(
    title='Permutation Importance: Feature × Model (F1 Macro)',
    template='plotly_white', height=400
)
show_and_save(fig_imp_heat, "importance_heatmap_all_models")

22:28:24 | Chromium init'ed with kwargs {}
22:28:24 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:28:24 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpuzlhx7gl.
22:28:24 | Opening browser.
22:28:24 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpcjsa61ae.
22:28:24 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpcjsa61ae
22:28:24 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpuzlhx7gl/index.html
22:28:24 | Waiting on all navigates
22:28:24 | All navigates done, putting them all in queue.
22:28:25 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpuzlhx7gl/index.html
22:28:25 | Waiting on all navigates
22:28:25 | All navigates done, putting them all in queue.
22:28:25 | Tab ready: 4920B11A8E40AA03A1319B6C9266738D
22:28:25 | Getting tab from queue (has 1)
22:28:25 | Got 4920
22:28:25 | Processing Permuta

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/importance_heatmap_all_models.png


## 8.3 Correlation vs. Causality — The Confounder Plot

Compares univariate correlation (descriptive) with multivariate importance (predictive). Features in the **upper left quadrant** (high correlation, low importance) are "confounders" — they correlate because they are proxies for true drivers.

In [25]:
# --- Correlation vs. Causality: Scatter Plot ---
df_combined = df_corr.merge(df_imp_avg, on='Feature')

# Feature classification
def classify_feature(row):
    corr_threshold = df_combined['Correlation'].median()
    imp_threshold = 0.005  # minimal positive importance
    if row['Correlation'] >= corr_threshold and row['Importance_mean'] >= imp_threshold:
        return '✅ True driver'
    elif row['Correlation'] >= corr_threshold and row['Importance_mean'] < imp_threshold:
        return '⚠️ Confounder'
    elif row['Correlation'] < corr_threshold and row['Importance_mean'] >= imp_threshold:
        return '🔍 Hidden driver'
    else:
        return '❌ Irrelevant'

df_combined['Category'] = df_combined.apply(classify_feature, axis=1)

# Scatter: Correlation (x) vs. Importance (y)
fig_scatter = px.scatter(
    df_combined, 
    x='Correlation', y='Importance_mean',
    text='Feature',
    color='Category',
    color_discrete_map={
        '✅ True driver': '#2ecc71',
        '⚠️ Confounder': '#e67e22',
        '🔍 Hidden driver': '#3498db',
        '❌ Irrelevant': '#95a5a6',
    },
    error_y='Importance_std',
    title='Correlation vs. Predictive Importance — "Confounder Plot"',
    labels={
        'Correlation': 'Univariate Correlation (Cramér\'s V / Eta²)',
        'Importance_mean': 'Permutation Importance (mean across models, F1 Macro)'
    },
    template='plotly_white'
)

# Quadrant lines
corr_median = df_combined['Correlation'].median()
fig_scatter.add_hline(y=0.005, line_dash="dash", line_color="grey", opacity=0.5,
                       annotation_text="Importance threshold")
fig_scatter.add_vline(x=corr_median, line_dash="dash", line_color="grey", opacity=0.5,
                       annotation_text="Correlation median")
fig_scatter.add_hline(y=0, line_dash="solid", line_color="black", opacity=0.3)

fig_scatter.update_traces(textposition='top center', marker=dict(size=14))
fig_scatter.update_layout(height=600, width=900)
show_and_save(fig_scatter, "correlation_vs_causality_scatter", width=1100, height=600)


06:47:33 | TemporaryDirectory.cleanup() worked.
06:47:33 | shutil.rmtree worked.
06:47:33 | Chromium init'ed with kwargs {}
06:47:33 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
06:47:33 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp051imatx.
06:47:33 | Opening browser.
06:47:33 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp8lw6vip0.
06:47:33 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp8lw6vip0
06:47:33 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp051imatx/index.html
06:47:33 | Waiting on all navigates
06:47:33 | All navigates done, putting them all in queue.
06:47:34 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp051imatx/index.html
06:47:34 | Waiting on all navigates
06:47:34 | All navigates done, putting them all in queue.
06:47:34 | Tab ready: 2CD2E6A9BC44AA1FC2086772D916F1E0
06:47:34 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/correlation_vs_causality_scatter.png


In [24]:
df_side = df_combined.sort_values('Correlation', ascending=True)

fig_side = go.Figure()
fig_side.add_trace(go.Bar(
    y=df_side['Feature'], x=df_side['Correlation'],
    name='Correlation (univariate)', orientation='h',
    marker_color='#3498db', opacity=0.8
))
fig_side.add_trace(go.Bar(
    y=df_side['Feature'], x=df_side['Importance_mean'],
    name='Importance (multivariate)', orientation='h',
    marker_color='#e74c3c', opacity=0.8,
    error_x=dict(type='data', array=df_side['Importance_std'].values, visible=True)
))
fig_side.update_layout(
    barmode='group',
    title='Correlation vs. Predictive Importance (Side-by-Side)',
    xaxis_title='Strength',
    yaxis_title=None,
    template='plotly_white',
    height=500,
    legend=dict(x=0.6, y=0.05)
)
show_and_save(fig_side, "correlation_vs_importance_bars")

log.info("\n📊 Feature Classification:\n")
display_df = df_combined[['Feature', 'Correlation', 'Method', 'Importance_mean', 'Importance_std', 'Category']]
display_df = display_df.sort_values('Correlation', ascending=False)
log.info(display_df.to_string(index=False, float_format='{:.4f}'.format))

df_combined.to_csv(RESULTS_DIR / "correlation_vs_importance.csv", index=False)

log.info("\n" + "="*60)
true_drivers = df_combined[df_combined['Category'].str.contains('True driver')]
confounders  = df_combined[df_combined['Category'].str.contains('Confounder')]
irrelevant   = df_combined[df_combined['Category'].str.contains('Irrelevant')]

if not true_drivers.empty:
    log.info(f"✅ True drivers: {', '.join(true_drivers['Feature'].tolist())}")
if not confounders.empty:
    log.info(f"⚠️  Confounders: {', '.join(confounders['Feature'].tolist())}")
    log.info(f"   → Correlate with target but no unique predictive power.")
if not irrelevant.empty:
    log.info(f"❌ Irrelevant: {', '.join(irrelevant['Feature'].tolist())}")
log.info("="*60)

22:28:28 | Chromium init'ed with kwargs {}
22:28:28 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:28:28 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpjv6vgl6_.
22:28:28 | Opening browser.
22:28:28 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpxl1afb22.
22:28:28 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpxl1afb22
22:28:28 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpjv6vgl6_/index.html
22:28:28 | Waiting on all navigates
22:28:28 | All navigates done, putting them all in queue.
22:28:28 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpjv6vgl6_/index.html
22:28:28 | Waiting on all navigates
22:28:29 | All navigates done, putting them all in queue.
22:28:29 | Tab ready: 04864F79D589A3AA7E6140E8F66B0881
22:28:29 | Getting tab from queue (has 1)
22:28:29 | Got 0486
22:28:29 | Processing Correla

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_221722/plots/correlation_vs_importance_bars.png


22:28:30 | 
📊 Feature Classification:

22:28:30 |        Feature  Correlation     Method  Importance_mean  Importance_std        Category
admission_type       0.1410 Cramér's V           0.0432          0.0036   ✅ True driver
     insurance       0.1235 Cramér's V           0.0037          0.0038   ⚠️ Confounder
          race       0.0843 Cramér's V           0.0014          0.0017   ⚠️ Confounder
marital_status       0.0665 Cramér's V          -0.0003          0.0008   ⚠️ Confounder
      language       0.0534 Cramér's V           0.0003          0.0016    ❌ Irrelevant
    anchor_age       0.0483       Eta²           0.0382          0.0104 🔍 Hidden driver
        gender       0.0395 Cramér's V           0.0007          0.0010    ❌ Irrelevant
           bmi       0.0089       Eta²           0.0143          0.0107 🔍 Hidden driver
22:28:30 | 
22:28:30 | ✅ True drivers: admission_type
22:28:30 | ⚠️  Confounders: insurance, race, marital_status
22:28:30 |    → Correlate with target but no

# Summary

This notebook provides four key results for the thesis:

1. **Benchmark** (Section 3-6): TabPFN compared against DummyClassifier, LogisticRegression, RandomForest, and XGBoost — all with identical data pipeline, 20 runs each.

2. **Data Quality** (Section 7): Quantifies how many target=2 patients are actually deceased (censored correctly, but important for interpretation).

3. **Korrelation vs. Kausalität** (Section 8.1–8.3): Systematic analysis distinguishing:
   - **Univariate Korrelation** (Cramér's V / Eta²): deskriptive Assoziation
   - **Permutation Importance** (F1 Macro, alle Modelle): prädiktive Bedeutung
   - **Hosenträger-Plot**: Identifiziert Confounder (hohe Korrelation, keine eigene Vorhersagekraft)


All results are saved in the run folder for reproducibility.